# Smart Healthcare Monitoring Platform

**Data types used:**
- Nominal: Patients.blood_type (categories like A+, O-, etc.)
- Ordinal: Vitals_Log.triage_alert_level (1=Low, 2=Medium, 3=Critical)
- Interval: Vitals_Log.body_temp_c (Celsius, arbitrary zero point)
- Ratio: Vitals_Log.heart_rate_bpm (true zero = no heartbeat)

In [4]:
import sqlite3
import random
import os
from datetime import datetime, timedelta
from faker import Faker

# settings
DB_NAME = "healthcare_monitoring.db"
NUM_PATIENTS = 50
NUM_DEVICES = 20
NUM_VITALS = 1050       # base vitals rows before duplicates
NULL_PERCENT = 0.03     # roughly 3% of temps will be set to NULL
NUM_DUPLICATES = 15     # number of duplicate rows to inject
SEED = 42

random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

# delete old db if it exists so we start fresh every time
if os.path.exists(DB_NAME):
    os.remove(DB_NAME)

conn = sqlite3.connect(DB_NAME)
cur = conn.cursor()

# turn on foreign key support (its off by default in sqlite)
cur.execute("PRAGMA foreign_keys = ON;")
print("Connected to database.")

Connected to database.


## Table Creation

In [5]:
# Patients table holds basic demographics
# blood_type is nominal data (unordered categories)
cur.execute("""
    CREATE TABLE IF NOT EXISTS Patients (
        patient_id  INTEGER PRIMARY KEY AUTOINCREMENT,
        first_name  TEXT NOT NULL,
        last_name   TEXT NOT NULL,
        email       TEXT UNIQUE,
        blood_type  TEXT
    );
""")

# Devices table stores info about each IoT medical device
cur.execute("""
    CREATE TABLE IF NOT EXISTS Devices (
        device_id         INTEGER PRIMARY KEY AUTOINCREMENT,
        device_type       TEXT NOT NULL,
        firmware_version  TEXT NOT NULL
    );
""")

# Patient_Device_Assignments maps patients to devices (many to many)
# This uses a composite primary key (patient_id + device_id)
cur.execute("""
    CREATE TABLE IF NOT EXISTS Patient_Device_Assignments (
        patient_id     INTEGER NOT NULL,
        device_id      INTEGER NOT NULL,
        assigned_date  DATE,
        PRIMARY KEY (patient_id, device_id),
        FOREIGN KEY (patient_id) REFERENCES Patients(patient_id),
        FOREIGN KEY (device_id)  REFERENCES Devices(device_id)
    );
""")

# Device_Maintenance is an audit log for maintenance on each device
# Second composite primary key (device_id + maintenance_date)
cur.execute("""
    CREATE TABLE IF NOT EXISTS Device_Maintenance (
        device_id          INTEGER NOT NULL,
        maintenance_date   DATE    NOT NULL,
        technician_notes   TEXT,
        PRIMARY KEY (device_id, maintenance_date),
        FOREIGN KEY (device_id) REFERENCES Devices(device_id)
    );
""")

# Vitals_Log is the main data table with 1000+ rows
# body_temp_c is interval data, heart_rate_bpm is ratio data,
# triage_alert_level is ordinal data (1=low, 2=medium, 3=critical)
# CHECK constraints keep values in realistic clinical ranges
cur.execute("""
    CREATE TABLE IF NOT EXISTS Vitals_Log (
        log_id              INTEGER  PRIMARY KEY AUTOINCREMENT,
        patient_id          INTEGER  NOT NULL,
        timestamp           DATETIME NOT NULL,
        body_temp_c         REAL     CHECK(body_temp_c BETWEEN 30.0 AND 45.0),
        heart_rate_bpm      INTEGER  NOT NULL CHECK(heart_rate_bpm BETWEEN 0 AND 250),
        triage_alert_level  INTEGER  NOT NULL CHECK(triage_alert_level IN (1, 2, 3)),
        FOREIGN KEY (patient_id) REFERENCES Patients(patient_id)
    );
""")

print("All 5 tables created.")

All 5 tables created.


## Inserting Data

In [6]:
# generate 50 patients with fake names and emails
blood_types = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]

patients = []
for i in range(NUM_PATIENTS):
    first = fake.first_name()
    last = fake.last_name()
    email = fake.unique.email()
    blood = random.choice(blood_types)
    patients.append((first, last, email, blood))

cur.executemany(
    "INSERT INTO Patients (first_name, last_name, email, blood_type) VALUES (?, ?, ?, ?);",
    patients
)
print(f"Inserted {NUM_PATIENTS} patients.")

Inserted 50 patients.


In [7]:
# generate 20 devices with random types and firmware versions
device_types = [
    "Heart Rate Monitor", "Pulse Oximeter", "Blood Pressure Cuff",
    "Thermometer Patch", "ECG Wearable", "Glucose Monitor",
    "Respiration Sensor", "Sleep Tracker"
]
firmwares = ["v1.0.0", "v1.1.2", "v1.2.5", "v2.0.0", "v2.1.0", "v3.0.0-beta"]

devices = []
for i in range(NUM_DEVICES):
    devices.append((random.choice(device_types), random.choice(firmwares)))

cur.executemany(
    "INSERT INTO Devices (device_type, firmware_version) VALUES (?, ?);",
    devices
)
print(f"Inserted {NUM_DEVICES} devices.")

Inserted 20 devices.


In [8]:
# assign each patient 1 to 3 devices
# using a set to avoid duplicate composite key combos
assigned = set()

for pid in range(1, NUM_PATIENTS + 1):
    count = random.randint(1, 3)
    chosen_devices = random.sample(range(1, NUM_DEVICES + 1), count)
    for did in chosen_devices:
        if (pid, did) not in assigned:
            assigned.add((pid, did))
            adate = fake.date_between(start_date="-1y", end_date="today")
            cur.execute(
                "INSERT INTO Patient_Device_Assignments (patient_id, device_id, assigned_date) VALUES (?, ?, ?);",
                (pid, did, adate.isoformat())
            )

print(f"Inserted {len(assigned)} patient-device assignments.")

Inserted 101 patient-device assignments.


In [9]:
# give each device 1 to 4 maintenance records
notes_list = [
    "Routine calibration completed.",
    "Battery replaced; firmware updated.",
    "Sensor cleaned; readings re-verified.",
    "Minor fault detected, replaced sensor module.",
    "Scheduled inspection; no issues found.",
    "Device returned from patient; factory reset applied.",
    "Wireless antenna repaired; connectivity test passed.",
    "Software patch applied for data-sync bug."
]

maint_set = set()
for did in range(1, NUM_DEVICES + 1):
    count = random.randint(1, 4)
    for _ in range(count):
        mdate = fake.date_between(start_date="-1y", end_date="today")
        # make sure composite key is unique
        if (did, mdate.isoformat()) not in maint_set:
            maint_set.add((did, mdate.isoformat()))
            note = random.choice(notes_list)
            cur.execute(
                "INSERT INTO Device_Maintenance (device_id, maintenance_date, technician_notes) VALUES (?, ?, ?);",
                (did, mdate.isoformat(), note)
            )

print(f"Inserted {len(maint_set)} maintenance records.")

Inserted 47 maintenance records.


In [10]:
# generate 1050 vital sign readings
# temperature uses a gaussian around 37.0 C so most values fall in the
# normal range of 36.5 to 37.5 with some outliers for fevers
# heart rate uses a gaussian around 75 bpm (resting range 60-100)
# triage level is weighted so most are low, fewer medium, rare critical
start_date = datetime.now() - timedelta(days=90)

vitals = []
for i in range(NUM_VITALS):
    pid = random.randint(1, NUM_PATIENTS)

    # random time in last 90 days
    offset = random.randint(0, 90 * 24 * 3600)
    ts = start_date + timedelta(seconds=offset)

    # temperature gaussian, clamped to stay within CHECK bounds
    temp = round(random.gauss(37.0, 0.4), 1)
    temp = max(30.0, min(45.0, temp))

    # heart rate gaussian, clamped to CHECK bounds
    hr = int(random.gauss(75, 12))
    hr = max(0, min(250, hr))

    # weighted triage: 70% low, 22% medium, 8% critical
    triage = random.choices([1, 2, 3], weights=[70, 22, 8], k=1)[0]

    vitals.append((pid, ts.strftime("%Y-%m-%d %H:%M:%S"), temp, hr, triage))

cur.executemany(
    "INSERT INTO Vitals_Log (patient_id, timestamp, body_temp_c, heart_rate_bpm, triage_alert_level) VALUES (?, ?, ?, ?, ?);",
    vitals
)
print(f"Inserted {NUM_VITALS} vitals rows.")

Inserted 1050 vitals rows.


## Deliberate Anomalies

In [11]:
# 1) Introducing missing data (NULLs)
# In real healthcare IoT systems, sensors can drop out temporarily.
# For example a wireless thermometer patch might lose bluetooth
# connection for a moment, so the temperature reading is missing
# but heart rate from a different sensor still comes through.
# We set about 3% of body_temp_c to NULL to simulate this.

cur.execute("SELECT log_id FROM Vitals_Log;")
all_ids = [row[0] for row in cur.fetchall()]

num_nulls = int(len(all_ids) * NULL_PERCENT)
null_ids = random.sample(all_ids, num_nulls)

for lid in null_ids:
    cur.execute("UPDATE Vitals_Log SET body_temp_c = NULL WHERE log_id = ?;", (lid,))

print(f"Set {num_nulls} temperature values to NULL (simulating sensor dropouts).")

Set 31 temperature values to NULL (simulating sensor dropouts).


In [12]:
# 2) Introducing duplicate rows
# In real IoT networks, if a device sends a reading but doesnt get
# an acknowledgement back in time, it retransmits the same packet.
# The server then ends up with duplicate entries for the same reading.
# We pick 15 random rows and re-insert them to simulate this.

dup_ids = random.sample(all_ids, NUM_DUPLICATES)

for lid in dup_ids:
    cur.execute(
        "SELECT patient_id, timestamp, body_temp_c, heart_rate_bpm, triage_alert_level "
        "FROM Vitals_Log WHERE log_id = ?;",
        (lid,)
    )
    row = cur.fetchone()
    cur.execute(
        "INSERT INTO Vitals_Log (patient_id, timestamp, body_temp_c, heart_rate_bpm, triage_alert_level) "
        "VALUES (?, ?, ?, ?, ?);",
        row
    )

print(f"Inserted {NUM_DUPLICATES} duplicate rows (simulating network retransmissions).")

conn.commit()

Inserted 15 duplicate rows (simulating network retransmissions).


## Verification

In [ ]:
# show all tables
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = [r[0] for r in cur.fetchall()]
print(f"Tables in database: {', '.join(tables)}")

# row counts
print("\nRow counts:")
for t in tables:
    cur.execute(f"SELECT COUNT(*) FROM [{t}];")
    print(f"  {t}: {cur.fetchone()[0]} rows")

# count nulls
cur.execute("SELECT COUNT(*) FROM Vitals_Log WHERE body_temp_c IS NULL;")
print(f"\nNULL body_temp_c values: {cur.fetchone()[0]}")

# count duplicates
cur.execute("""
    SELECT patient_id, timestamp, body_temp_c, heart_rate_bpm, triage_alert_level,
           COUNT(*) AS cnt
    FROM Vitals_Log
    GROUP BY patient_id, timestamp, body_temp_c, heart_rate_bpm, triage_alert_level
    HAVING COUNT(*) > 1;
""")
dups = cur.fetchall()
print(f"Duplicate row groups: {len(dups)}")

# show a few sample rows
print("\nSample patients:")
cur.execute("SELECT * FROM Patients LIMIT 3;")
for r in cur.fetchall():
    print(f"  {r}")

print("\nSample vitals:")
cur.execute("SELECT * FROM Vitals_Log LIMIT 3;")
for r in cur.fetchall():
    print(f"  {r}")

conn.close()

Tables in database: Device_Maintenance, Devices, Patient_Device_Assignments, Patients, Vitals_Log, sqlite_sequence

Row counts:
  Device_Maintenance: 47 rows
  Devices: 20 rows
  Patient_Device_Assignments: 101 rows
  Patients: 50 rows
  Vitals_Log: 1065 rows
  sqlite_sequence: 3 rows

NULL body_temp_c values: 31
Duplicate row groups: 15

Sample patients:
  (1, 'Danielle', 'Johnson', 'john21@example.net', 'A-')
  (2, 'Joy', 'Gardner', 'fjohnson@example.org', 'A+')
  (3, 'Jesse', 'Guzman', 'jennifermiles@example.com', 'AB+')

Sample vitals:
  (1, 23, '2026-02-26 00:48:56', 37.3, 78, 1)
  (2, 29, '2026-02-21 11:01:11', 36.7, 72, 3)
  (3, 42, '2026-01-22 20:46:28', 37.7, 76, 1)

Done. Database saved as healthcare_monitoring.db
